In [46]:
import numpy as np

IS_help = np.load("/kaggle/input/datasets/ahmedorapi/hello-stop-help/last_Help.npy")

Help = IS_help.transpose(1, 0)

print(Help.shape)

(19, 48)


In [47]:
import numpy as np

IS_hello = np.load("/kaggle/input/datasets/ahmedorapi/hello-stop-help/last_Hello.npy")

Hello = IS_hello.transpose(1, 0)
 
print(Hello.shape)

(19, 48)


In [48]:
import numpy as np

IS_stop = np.load("/kaggle/input/datasets/ahmedorapi/hello-stop-help/last_Stop.npy")

Stop = IS_stop.transpose(1, 0)

print(Stop.shape)

(19, 48)


In [49]:
digit_data = np.load("/kaggle/input/datasets/ahmedorapi/daigits/sampled_correct_predictions.npz")
print(digit_data.files)

d0 = digit_data['X'][0]
d1 = digit_data['X'][11]
d4 = digit_data['X'][21]
d5 = digit_data['X'][31]

['X', 'y', 'preds']


In [50]:
digit_data = np.load("/kaggle/input/datasets/ahmedorapi/motor-imagery/proccesed_correct_5per_class.npz")
print(digit_data.files)

#   0 = Left fist   
#   1 = Right fist  
#   2 = Both fists  
#   3 = Both feet   
Left_fist = digit_data['X'][0]
Right_fist = digit_data['X'][6]
Both_fists = digit_data['X'][13]
Both_feet = digit_data['X'][10]

['X', 'y']


In [51]:
n_MI     = len(Left_fist[1])   #641
n_IS     = len(Hello[1])     #48
n_digits = len(d0[1])          #256


In [52]:
def make_seq(seq):
    sequence = []
    MI_seq = []
    IS_seq = []
    digits_seq = []
    
    data_dict = {
    "Left_fist": Left_fist,
    "Right_fist": Right_fist,
    "Both_fists": Both_fists,
    "Both_feet": Both_feet,
    "Hello": Hello,
    "Help": Help,
    "Stop": Stop,
    "d0": d0,
    "d1": d1,
    "d4": d4,
    "d5": d5,
}

    
    c_MI = ['Left_fist', 'Right_fist', 'Both_fists', 'Both_feet']
    c_IS = ['Hello', 'Help', 'Stop']
    c_digits = ['d0','d1', 'd4', 'd5']
    
    ch_MI, n_MI,t_IM, f_MI    = 44, 641, 4, 160
    ch_IS, n_IS, t_IS, f_IS    = 19, 48, 2, 24
    ch_digits, n_digits, t_digits, f_digits = 14, 256, 2, 128


    MI_seq = np.empty((ch_MI, 0), dtype=np.float32)
    IS_seq = np.empty((ch_IS, 0), dtype=np.float32)
    digits_seq = np.empty((ch_digits, 0), dtype=np.float32)

    def make_zeros(channel, f, t):
        return np.zeros((channel, int(f * t)), dtype=np.float32)
 

    for command in seq:
        if command in c_MI:
            MI_seq = np.concatenate([MI_seq, data_dict[command]], axis=1)
            IS_seq = np.concatenate([IS_seq, make_zeros(ch_IS, f_IS, t_IM)], axis=1)
            digits_seq = np.concatenate([digits_seq, make_zeros(ch_digits, f_digits, t_IM)], axis=1)
        
        elif command in c_IS:
            IS_seq = np.concatenate([IS_seq, data_dict[command]], axis=1)
            MI_seq = np.concatenate([MI_seq, make_zeros(ch_MI, f_MI, t_IS)], axis=1)
            digits_seq = np.concatenate([digits_seq, make_zeros(ch_digits, f_digits, t_IS)], axis=1)
        
        elif command in c_digits:
            digits_seq = np.concatenate([digits_seq, data_dict[command]], axis=1)
            IS_seq = np.concatenate([IS_seq, make_zeros(ch_IS, f_IS, t_digits)], axis=1)
            MI_seq = np.concatenate([MI_seq, make_zeros(ch_MI, f_MI, t_digits)], axis=1)

    return MI_seq, IS_seq, digits_seq
                  
    



In [53]:
mapping_dict = {
    "Register":"Left_fist",
    "Forget":"Right_fist",
    "Collection":"Both_fists",
    "Device":"Both_feet",
    "Hello":"Hello",
    "Discover":"Help",
    "End":"Stop",
    "0":"d0",
    "1":"d1",
    "4":"d4",
    "5":"d5"
}

In [54]:
seq = ['Discover', 'Collection', '4']

mapped_seq = [mapping_dict[comm] for comm in seq]    

MI_seq, IS_seq, digits_seq = make_seq(mapped_seq)

np.save("MI_seq.npy", MI_seq)
np.save("IS_seq.npy", IS_seq)
np.save("digits_seq.npy", digits_seq)

print("Saved successfully")

Saved successfully


In [55]:
MI = np.load('/kaggle/working/MI_seq.npy')
print(MI.shape)
IS = np.load('/kaggle/working/IS_seq.npy')
print(IS.shape)
digits = np.load('/kaggle/working/digits_seq.npy')
print(digits.shape)

(44, 1281)
(19, 192)
(14, 1024)
